# DEMO-01 — Synthetic Demo Batch Generator

Generates **500,000 realistic taxi trips** spread across **all 30 days of April 2016** (outside the original dataset range) and appends them to Bronze.  
This lets us demonstrate live data flowing through Bronze → Silver → Gold → Dashboard during the stakeholder demo.

**Schema:** Matches `students_data.chris-foreman.bronze_yellow_tripdata` exactly (19 columns + `_rescued_data` + `_ingested_at`).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType,
    TimestampType, StringType,
)

# Must match Bronze table exactly
BRONZE_TABLE = "students_data.`chris-foreman`.bronze_yellow_tripdata"

# Demo batch parameters
NUM_TRIPS = 500_000
DEMO_START = "2016-04-01"
DEMO_END = "2016-04-30"
SEED = 42

In [0]:
# ---------------------------------------------------------------------------
# 500K trips generated entirely with PySpark-native functions (no Python loop)
# ---------------------------------------------------------------------------

# Start with a range DataFrame
raw = spark.range(NUM_TRIPS).withColumn("_r", F.rand(seed=SEED))

# ── Pickup datetime: random day in April 2016 + random time ──────────────
raw = (
    raw
    .withColumn("day_offset", (F.rand(seed=SEED + 1) * 30).cast("int"))  # 0-29
    .withColumn("hour", F.floor(F.rand(seed=SEED + 2) * 24).cast("int"))
    .withColumn("minute", F.floor(F.rand(seed=SEED + 3) * 60).cast("int"))
    .withColumn("second", F.floor(F.rand(seed=SEED + 4) * 60).cast("int"))
    .withColumn(
        "tpep_pickup_datetime",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_add(F.lit(DEMO_START), F.col("day_offset")),
                F.concat_ws(
                    ":",
                    F.lpad(F.col("hour"), 2, "0"),
                    F.lpad(F.col("minute"), 2, "0"),
                    F.lpad(F.col("second"), 2, "0"),
                ),
            )
        ),
    )
)

# ── Trip duration: 4-45 min skewed towards ~12 min (triangular approx) ───
# Approximate triangular(4, 45, 12) using: mode + spread via two uniform draws
raw = raw.withColumn(
    "duration_min",
    F.round(
        F.lit(4.0) + (F.lit(41.0) * F.least(F.rand(seed=SEED + 5), F.rand(seed=SEED + 6))
        * F.lit(0.8) + F.rand(seed=SEED + 7) * F.lit(41.0) * F.lit(0.2)),
        1,
    ),
).withColumn("duration_min", F.when(F.col("duration_min") > 45, F.lit(45.0)).otherwise(F.col("duration_min")))

# Dropoff = pickup + duration
raw = raw.withColumn(
    "tpep_dropoff_datetime",
    (F.col("tpep_pickup_datetime").cast("double") + F.col("duration_min") * 60).cast("timestamp"),
)

# ── Pickup / dropoff locations: 10 NYC landmarks with jitter ────────────
# Encode landmarks as arrays and pick randomly
landmark_lats = [40.7580, 40.6413, 40.7506, 40.7527, 40.7769,
                 40.7074, 40.7648, 40.7061, 40.7736, 40.7233]
landmark_lons = [-73.9855, -73.7781, -73.9935, -73.9772, -73.8740,
                 -74.0113, -73.9724, -73.9969, -73.9566, -73.9985]

lat_arr = F.array([F.lit(v) for v in landmark_lats])
lon_arr = F.array([F.lit(v) for v in landmark_lons])

raw = (
    raw
    .withColumn("pickup_idx", F.floor(F.rand(seed=SEED + 10) * 10).cast("int"))
    .withColumn("dropoff_idx", F.floor(F.rand(seed=SEED + 11) * 10).cast("int"))
    # Ensure pickup != dropoff
    .withColumn(
        "dropoff_idx",
        F.when(F.col("dropoff_idx") == F.col("pickup_idx"), (F.col("dropoff_idx") + 1) % 10)
        .otherwise(F.col("dropoff_idx")),
    )
    .withColumn("pickup_latitude",   lat_arr[F.col("pickup_idx")]  + (F.rand(seed=SEED + 12) - 0.5) * 0.01)
    .withColumn("pickup_longitude",  lon_arr[F.col("pickup_idx")]  + (F.rand(seed=SEED + 13) - 0.5) * 0.01)
    .withColumn("dropoff_latitude",  lat_arr[F.col("dropoff_idx")] + (F.rand(seed=SEED + 14) - 0.5) * 0.01)
    .withColumn("dropoff_longitude", lon_arr[F.col("dropoff_idx")] + (F.rand(seed=SEED + 15) - 0.5) * 0.01)
)

# ── Trip distance: loosely correlated with duration ──────────────────────
raw = raw.withColumn(
    "trip_distance",
    F.round(F.lit(0.5) + F.col("duration_min") / 4.0 * F.rand(seed=SEED + 20) * 2.0, 2),
)

# ── Weighted categorical columns ────────────────────────────────────────
# VendorID: 1 or 2 (50/50)
raw = raw.withColumn("VendorID", F.when(F.rand(seed=SEED + 30) < 0.5, 1).otherwise(2).cast("long"))

# passenger_count: weighted 65/15/8/5/4/3 for 1-6
r_pax = F.rand(seed=SEED + 31)
raw = raw.withColumn(
    "passenger_count",
    F.when(r_pax < 0.65, 1)
    .when(r_pax < 0.80, 2)
    .when(r_pax < 0.88, 3)
    .when(r_pax < 0.93, 4)
    .when(r_pax < 0.97, 5)
    .otherwise(6)
    .cast("long"),
)

# RateCodeID: weighted 90/5/2/2/1 for 1-5
r_rate = F.rand(seed=SEED + 32)
raw = raw.withColumn(
    "RateCodeID",
    F.when(r_rate < 0.90, 1)
    .when(r_rate < 0.95, 2)
    .when(r_rate < 0.97, 3)
    .when(r_rate < 0.99, 4)
    .otherwise(5)
    .cast("long"),
)

# store_and_fwd_flag: 95% N, 5% Y
raw = raw.withColumn(
    "store_and_fwd_flag",
    F.when(F.rand(seed=SEED + 33) < 0.95, F.lit("N")).otherwise(F.lit("Y")),
)

# payment_type: 70% credit (1), 30% cash (2)
raw = raw.withColumn(
    "payment_type",
    F.when(F.rand(seed=SEED + 34) < 0.70, 1).otherwise(2).cast("long"),
)

# ── Fare breakdown ──────────────────────────────────────────────────────
raw = (
    raw
    .withColumn("fare_amount", F.round(F.lit(2.50) + F.col("trip_distance") * 2.50 + F.col("duration_min") * 0.40, 2))
    .withColumn(
        "extra",
        F.when(F.rand(seed=SEED + 40) < 0.33, 0.0)
        .when(F.rand(seed=SEED + 41) < 0.50, 0.5)
        .otherwise(1.0),
    )
    .withColumn("mta_tax", F.lit(0.50))
    .withColumn("improvement_surcharge", F.lit(0.30))
    # Tips: 0-25% of fare for credit card only
    .withColumn(
        "tip_amount",
        F.when(
            F.col("payment_type") == 1,
            F.round(F.col("fare_amount") * F.rand(seed=SEED + 42) * 0.25, 2),
        ).otherwise(0.0),
    )
    # Tolls: ~10% of trips at $5.54
    .withColumn(
        "tolls_amount",
        F.when(F.rand(seed=SEED + 43) < 0.10, 5.54).otherwise(0.0),
    )
    .withColumn(
        "total_amount",
        F.round(
            F.col("fare_amount") + F.col("extra") + F.col("mta_tax")
            + F.col("tip_amount") + F.col("tolls_amount") + F.col("improvement_surcharge"),
            2,
        ),
    )
)

# ── Metadata columns ────────────────────────────────────────────────────
raw = (
    raw
    .withColumn("_rescued_data", F.lit(None).cast("string"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# ── Select final columns in exact Bronze order ──────────────────────────
demo_df = raw.select(
    F.col("VendorID").cast("long"),
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    F.col("passenger_count").cast("long"),
    F.col("trip_distance").cast("double"),
    F.col("pickup_longitude").cast("double"),
    F.col("pickup_latitude").cast("double"),
    F.col("RateCodeID").cast("long"),
    "store_and_fwd_flag",
    F.col("dropoff_longitude").cast("double"),
    F.col("dropoff_latitude").cast("double"),
    F.col("payment_type").cast("long"),
    F.col("fare_amount").cast("double"),
    F.col("extra").cast("double"),
    F.col("mta_tax").cast("double"),
    F.col("tip_amount").cast("double"),
    F.col("tolls_amount").cast("double"),
    F.col("improvement_surcharge").cast("double"),
    F.col("total_amount").cast("double"),
    "_rescued_data",
    "_ingested_at",
)

print(f"✅ Generated {demo_df.count():,} synthetic trips for April 2016")

In [0]:
demo_df.display()

In [0]:
# Append-only write — mirrors the real ingestion pattern
demo_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

print(f"✅ Appended {NUM_TRIPS} demo trips to {BRONZE_TABLE}")

In [0]:
from pyspark.sql import functions as F

verify_df = (
    spark.read.table(BRONZE_TABLE)
    .filter(F.col("tpep_pickup_datetime").cast("date").between("2016-04-01", "2016-04-30"))
)

print(f"✅ Bronze rows for April 2016: {verify_df.count():,}")
verify_df.select("tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount").display()

In [0]:
# Uncomment to remove demo rows after the presentation
#from pyspark.sql import functions as F
#from delta.tables import DeltaTable
#dt = DeltaTable.forName(spark, BRONZE_TABLE)
#dt.delete(F.col("tpep_pickup_datetime").cast("date").between("2016-04-01", "2016-04-30"))
#print("🧹 Demo rows for April 2016 removed from Bronze")